# Run Kubernetes experiment bundles

First generate configurations with `generate_simulation_files.ipynb` and build/load
all referenced images following `../kubernetes-launcher/README.md`. The current
`kubectl` context selects the cluster and namespace. Run bundles sequentially:
the launcher uses fixed Kubernetes resource names and handles measurement and cleanup.

Relative configuration-folder paths below are relative to `kubernetes-experiments`,
whether the kernel starts here or at the workspace root. Each repetition gets a
unique run ID. Results, the effective config, and launcher logs are kept under
`results/<case>/<mode>/<run-id>/<simulation-name>/` for generated configurations.
The JSON Lines format is the launcher's format; the old Podman plotting notebook
has not been adapted to it.


In [1]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import re
import runpy
import signal
import subprocess
from uuid import uuid4

WORKSPACE_DIR = next(
    directory for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "kubernetes-launcher/launch.sh").is_file()
)
EXPERIMENTS_DIR = WORKSPACE_DIR / "kubernetes-experiments"
LAUNCH_SCRIPT = WORKSPACE_DIR / "kubernetes-launcher/launch.sh"
validate_config = runpy.run_path(
    str(LAUNCH_SCRIPT.with_name("render-experiment.py"))
)["validate_config"]


In [2]:
def new_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "-" + uuid4().hex[:8]


def experiment_paths(config_path, run_id):
    """Validate before launching and keep all bundle artifacts in this folder."""
    if not re.fullmatch(r"[a-zA-Z0-9_-]+", run_id):
        raise ValueError("run_id must contain only letters, digits, underscores or hyphens")
    config_path = Path(config_path).expanduser()
    if not config_path.is_absolute():
        config_path = EXPERIMENTS_DIR / config_path
    config_path = config_path.resolve()
    config = validate_config(config_path)
    output_base = Path(config.pop("results_file")).parent
    if not output_base.is_relative_to(EXPERIMENTS_DIR.resolve()):
        raise ValueError(
            f"{config_path}: output_path must resolve inside {EXPERIMENTS_DIR}"
        )
    output_dir = (output_base / run_id / config["simulation_name"]).resolve()
    if not output_dir.is_relative_to(EXPERIMENTS_DIR.resolve()):
        raise ValueError(f"Result directory escapes {EXPERIMENTS_DIR}: {output_dir}")
    config["output_path"] = str(output_dir)
    return config, output_dir


def run_experiment(config_path, *, run_id=None):
    """Run one JSON config; retain its measurements and logs even on failure."""
    run_id = run_id or new_run_id()
    config, output_dir = experiment_paths(config_path, run_id)
    # Refuse to overwrite artifacts if the same run ID is reused.
    output_dir.mkdir(parents=True, exist_ok=False)
    effective_config = output_dir / "experiment.json"
    effective_config.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
    print(f"Running {config['simulation_name']} ({run_id})", flush=True)

    with (output_dir / "launch.log").open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            ["bash", str(LAUNCH_SCRIPT), str(effective_config)],
            cwd=WORKSPACE_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except KeyboardInterrupt:
                # Forward notebook interrupts to the launcher and wait for its cleanup trap.
                try:
                    os.killpg(process.pid, signal.SIGINT)
                except ProcessLookupError:
                    pass  # The launcher may have exited just before the interrupt.
                process.wait()
                raise
    if returncode:
        raise subprocess.CalledProcessError(
            returncode, ["bash", str(LAUNCH_SCRIPT), str(effective_config)]
        )
    results_file = output_dir / f"{config['simulation_name']}-measurements.jsonl"
    if not results_file.is_file():
        raise FileNotFoundError(f"Launcher succeeded but no measurements found: {results_file}")
    return results_file


def run_experiments(config_folder, *, repetitions=1, recursive=False, dry_run=False):
    """Run sorted JSON files sequentially, stopping on the first failure.

    dry_run validates every configuration and prints destinations without launching.
    recursive=True includes nested folders, e.g. all of simulation-files.
    """
    if isinstance(repetitions, bool) or not isinstance(repetitions, int) or repetitions < 1:
        raise ValueError("repetitions must be a positive integer")
    directory = Path(config_folder).expanduser()
    if not directory.is_absolute():
        directory = EXPERIMENTS_DIR / directory
    if not directory.is_dir():
        raise FileNotFoundError(f"Configuration folder not found: {directory}")
    configs = sorted(directory.rglob("*.json") if recursive else directory.glob("*.json"))
    if not configs:
        raise ValueError(f"No JSON configurations found in {directory}")

    # Preflight the whole bundle before creating Kubernetes resources.
    plans = []
    destinations = set()
    for _ in range(repetitions):
        run_id = new_run_id()
        for config_path in configs:
            _, output_dir = experiment_paths(config_path, run_id)
            if output_dir in destinations or output_dir.exists():
                raise FileExistsError(f"Duplicate or existing result directory: {output_dir}")
            destinations.add(output_dir)
            plans.append((config_path, run_id, output_dir))
    if dry_run:
        for config_path, run_id, output_dir in plans:
            print(f"{config_path.name} ({run_id}) -> {output_dir}")
        return [output_dir for _, _, output_dir in plans]
    return [run_experiment(path, run_id=run_id) for path, run_id, _ in plans]


## Validate and preview a bundle

This cell contacts no cluster and creates no result files. Change `CONFIG_FOLDER`
to any folder of experiment JSON files. Their `output_path` must resolve inside
`kubernetes-experiments` (relative paths resolve from each config's directory).


In [ ]:
CONFIG_FOLDER = "simulation-files/good-case/both"
REPETITIONS = 1

planned_results = run_experiments(CONFIG_FOLDER, repetitions=REPETITIONS, dry_run=True)

commit-awset-12r-12c-64f.json (20261119T140549642262Z-85733461) -> /volatile/home/jb287061/Documents/MyPapers/DagOracleCommitment/Moirai/kubernetes-experiments/results/good-case/optimistic/20261119T140549642262Z-85733461/commit-awset-12r-12c-64f
commit-awset-4r-4c-64f.json (20261119T140549642262Z-85733461) -> /volatile/home/jb287061/Documents/MyPapers/DagOracleCommitment/Moirai/kubernetes-experiments/results/good-case/optimistic/20261119T140549642262Z-85733461/commit-awset-4r-4c-64f
commit-awset-8r-8c-64f.json (20261119T140549642262Z-85733461) -> /volatile/home/jb287061/Documents/MyPapers/DagOracleCommitment/Moirai/kubernetes-experiments/results/good-case/optimistic/20261119T140549642262Z-85733461/commit-awset-8r-8c-64f
vanilla-awset-12r-12c-64f.json (20261119T140549642262Z-85733461) -> /volatile/home/jb287061/Documents/MyPapers/DagOracleCommitment/Moirai/kubernetes-experiments/results/good-case/optimistic/20261119T140549642262Z-85733461/vanilla-awset-12r-12c-64f
vanilla-awset-4r-4c-64

## Run the selected bundle

Set `RUN_EXPERIMENTS = True` to execute. Each call creates fresh run IDs so repeated
executions preserve earlier results. A launcher failure stops the bundle and raises
an exception; its `launch.log`, effective `experiment.json`, and any partial
measurements remain in the result directory. Interrupting the cell signals the
launcher and waits for Kubernetes cleanup.


In [20]:
RUN_EXPERIMENTS = True

if RUN_EXPERIMENTS:
    result_files = run_experiments(CONFIG_FOLDER, repetitions=REPETITIONS)
    #print(*result_files, sep="\n")

Running commit-awset-12r-12c-128f (20261119T141400167882Z-3c7f29bb)
Experiment: commit-awset-12r-12c-128f
Replicas: 12; clients: 12
configmap/moirai-multi-bootstrap-k85h8f9m86 created
service/moirai-multi-http created
service/moirai-multi-replicas created
statefulset.apps/moirai-multi-replica created
Waiting for statefulset spec update to be observed...
Waiting for 12 pods to be ready...
Waiting for 12 pods to be ready...
Waiting for 11 pods to be ready...
Waiting for 10 pods to be ready...
Waiting for 9 pods to be ready...
Waiting for 8 pods to be ready...
Waiting for 7 pods to be ready...
Waiting for 6 pods to be ready...
Waiting for 5 pods to be ready...
Waiting for 4 pods to be ready...
Waiting for 3 pods to be ready...
Waiting for 2 pods to be ready...
Waiting for 1 pods to be ready...
partitioned roll out complete: 12 new pods have been updated...
job.batch/moirai-client-load created
Collected 25 result records into /volatile/home/jb287061/Documents/MyPapers/DagOracleCommitment/M

## Other bundles or a single configuration

Run these snippets after editing the selection above, or uncomment a call below.


In [5]:
# Faulty case, both request modes:
# result_files = run_experiments("simulation-files/faulty-case/both", repetitions=3)

# All generated cases and request modes:
# result_files = run_experiments("simulation-files", repetitions=3, recursive=True)

# One configuration:
# results_file = run_experiment(
#     "simulation-files/good-case/optimistic/commit-awset-4r-4c-256f.json"
# )
